# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "a94943eb4ce12d307715b010f4b5b6871314e4d52badde7de5a1e463d5990312",
    "gzip_base64": "H4sIAAAAAAAC/7U825LbRnbv8xWQ9gHADAhyRpZscQaq8jrazVaytsr2plLFsFBNoEkiAwI0AM7FiqryD/nDfEnOrYEGCFJyvKuHEdHoPt3n9Ln3abx8+fLb795Pfmr03lmVhyLV6SQpi0Y/NY5+2usq2+miqR1VpM5Oq/pQ6TRwarXb53qi8mxT6NSpm6xJtlmxCS8uftSHWtdOs9VOstXJPbxeqbqeptVhN90cskZVzmNZ3euqDp0PuSqm+6pcaSfVa3XIG6cpnaJ0smKtK10kOrz4Xuu0dorDbv8MEycZ/YcrXWe5Dpz1erfXm8DZ5weeVT9lsJ5i43z3t3/61qkORQMoOOuyciYT/aSTQwNAX758eZHt9mXVOKra7FVVa/O8VfU2z1bm8T/rsjC/d6rZXqyrcufs4Rd0cuTFB3whv+vtocny9umwAvwSXdcXpolQcRTgtG97GXywuV7zHIRrWAOJVW4mSsqq0rlqEPGsSOO9VvcdZJ3rpKmyJF4f8jyGhW8IHhAj17Az3/783omcVzez2ezixx9++BkecOFeHOPUceyHla7L/EF7fggkwX1fXC8vLi5gbwAtdfP6jYeI+/MLB/6l2UbXTST0CqWDT+8es2bLwGlAWO514bnVyvVpRU2l1Y6h4D/cnFVeJvew707W6MrL1W6Vqjl3hGWp1Lue3Xx1iX/8YOW6/pynDw/7FMjh0XCeu9LNoSpkeeFWP/EvWBlj8ljBFDHuKy0ueFD5QQtO2BB1675tNFJWVc8RPoeIVlwf1uvsyXPDZrd3ecq2W8jAUXw8nCFMYbNrj6YIYMOAptFNoPK8fIwLVUR/Unmt/Sv3P4ojSJXe5yrRvBBZeq6ey0PjNWWj8sAROY3ezgJgzWqTFdEb4AuV6ih8/Q28P+g6+r4sgFl0oavNMz0IprSkOvIESMAAAhzNK8nWjqeKZy8D/qkbBbLoPQSrssx9B7arKBun98bLiiZY56Vq/LYDykuY1cCpQBTvwaeNfsBN5tn9lgNkwPXsLpIF3UVvZjPTPrvDdd3xGrGRf91d428ZMLm5NK0zv+OtSmW1dv4N53tfVWXluahQzCCY8X//+39gptqQ8F10XTMRcZ3eTAjjI7krDW9ntdsSaEAF3hcghN8hdIlidxfRK8KJGs6t76fyUCXa2R3qxgHFaFbo1BoWnZrZ/+D8BTDYaKBFmercSQ+VarKyqGHZSX6A5WtQ2XuVpqiXne/1A3TVTwkij2qy0r/ADjRICrUPCeROPWW7wy6ibYOthMUImXxatEhIGlWorjwmDL3xb4k6kUCArYBuhkj45u7683gbfMzWwLqcXKsHsCY56E9YtSqAGrVTMkxHHdKsFHIksKQmusb5hNSyFlCKMA88eDcBIZboLPd4q3id/hSh+QJH7VWSNRmIxgKbr6ADAs2QGxazgOaZXC8ZLC0DmZpeV6rYaI96+MsOe1xYdD23IcvauBNYJGiaLW+dfZlnCXUwb+g3gYGfwGoOSrDMDS1mkoGcNifltPkCOW06sewkc3bX3BHNpmRGEOcGccZVEBz88SKqwQzp1Kt149Gbs4L4R2QjUHUOWui6ZXheRuAcigxYFI09wiQPBPDIgLORf9G6yd4PNuCaN8meWu8UAC02LIYTpPgCNrHtAE7JRjeRab9iBm+HXXZ7t8jg/bQ+7Lx+23zpd2TLS+Q+rwUnzB/w7MPBvaHbLNplRTd0MLOAYCabZP6lgO4AADfk5bttNh9RfKWzBhcuW4GfsTK0R5evGmoElsCp6Ns822WN0TsWuXCh2yxAVPMy4MYeFQLExn4LAkerDWQot17dDHAo96TJogXvQ8Ov+0zHiN5FvS530TZb2qQQSHObo1HgaO3yMrjXzxE7HE4zV6vaayYGGZHJ58gFL+4BeC+NDeXcFiaK42AGYNYM/ZI6ysEb9ZgxiSBX1wEudToFK0WCsxgn43J0yR3g/qo9NvAgu4g1P7CamPlX4Wx2c9nDa9qnN+l1wRM9Ex4fF1pVMY9wh2DdlcpRo6QxCm9HC2TcUO3B3UuRif1OqZlWnsi/GPYmzpbm7aG4b7UgiXfgqWDl4+Zr8J91hT7fr9meBCUgaSER7LYBFF7VBACYOHAWqAmqemI8NtMrVO0dDYz9jICU6YRGM5naHhDFaAkC2IiYIX7rtHRkIAwMah9ddP2e3HkWuGgyYgLvzpU8Qh93vgqg1/7QmJeMgLRRD/gT9PbM+ueSFxCbJbnzkWV+6vnHH12O4mKkpTsnBeUSaeJ1pUAhu3MmlCvIxaCLYuOEzI3feNFfBWmMrpd4lS76U10rPgWuEaTYcIg7N79gTiIgTEP/fxL/FwOBmMx+LxBRjQrqKqrXHCmQV6/yR/Vcxzdp9HMFViRtnvc6csnwvbrpPLi6ehGRRQMmQzgYxuw1hD0vohtj+4p9Zxyxjx+CA+993sfkIcCJEMqWEHfd//OvPZ/FBCoAUvBDOw1xMkQJcbn2wP8vZJbuRYStIYik53aNbtC17iGIA4ENPn7yj7u5GIu7fot+9449BfB4uUvgYsCO/3PMDgHXiKd62O/ZONM2WdDqOUX9gUNQAscAsfHuepvAjNICMci7170zBLDXGkW8yLlAkgh3rB9j0W2VRMrYHFPiwXISeDf6b4bwBJEjgNw+DvLo3d+FkkwzjNxrIpnFLa2EC/ae22EVYlhqtjd8VA/mNyUNXH8g0AZfz7WxMEC4TaBIh1EwvF2eyzsV7+HRwGiTFthHQPXaXP/TYkQwTGKC+X1IAualaJSlWhAjYtC+G9vrB1VlqieCgrO8gJWXVQYKT+VuzyOT90bGeFXhhx/f//T+559GBOtvxX1RPhYOToQJNsmbmWn8IYf1AS6k3/LiVJeOdJjn8WqKOAOnPDRgbL4ssUA5I8kuXAa2kIgCgYklh8caW/r2KRZZBPv8xvnnRJICBFn5u7ezEar+malox5fgT+GeSFz8dhhkw1T99b6wFmwiElnJi45hTk5t+ABd7Qz41lFma3GnZVJm6Mji64+2Dp93v4Mh/837z59aLAZ7gZEk4k3RZB+Nzg58iQHYAvukHfQOMRqMBrSn+YXRmM84z8ZNVuoxoGbuYjXTeG4Nd/dpBtLOKUq275T1jct7yahxb9mpGFOUkeQnZT7qsEGPlnyjmNysiOFPPfa63CvuHHLGLwTXWFeeaA1O84ZJuX9GSTAyNATZORo8/dH7F1Fvmeecih+ko0MjwVJgVJE6j1tMHLMsY94bYzmLnZJcq8Jg5gpGqGYljfSdKsoiSwBsyxrocZPqAXDTXfaEUMsifw6dP/PyIUjHDDXNhGxaPehUwBlsYK9pTZz8RgjgkSZbORrAqAhBTcBDhsAKzxDIMkzQMjhkY2pOSnUJ9LA6FN7C5YQ/8OWkKOsmzdCGTMhoIJWoHf9kbgAsbbYbGoAJqYUD0FE/GjolMPYGxydzBf/tk11cX7/JNY8lWvrLgE42iPGYxqzgLO+UO3ZJHBS0NiNktGe31zvdKHQDI8oZg4+a1h6JQdvXJ9eWE8t+z7aYsWyQGN+Yuc0F7uqzfT9MPZOSMVCdtNS1SRIlW9o8a4spU4l8a9kkQri/qhSM2UqrBtTJYulfDZYMth7jcHrXh3KUTmJ/X5tY1KNIxcKKjgEi2oMFx33UY8Ih9zf+HOO/XBcedfH57ZV5uxzaVQoXPPD/618q+n8H28pnDZeXN76sB0UlktS8BXsksx4gFgGvvRtrzjBqYKudih90VWMgegMsp5pDHbniBrhBqyp4V/uqI5C9J8NsC/oxs/e4xLAI82zQMz+BZWktu3ME0FJr9iqG2i4sgNzBcedRdI4mAS4ExL6X00E0PMBIupq23BU631m2XfLgtbgwtRg6TjL3kuN/gegcA3U0fxxw3pKXkBVgvJ757PNQk4785aBybDsUsE3ZOtNp6H7eLTntRMxxbxdDO76M+g1fYMbtbGff7yJFMmgcGlbKs1Tlr7robGBNxj1uR7pXAyCjdtE6GRS+Gk59OzSdgw4BL6Sn5ASUvHkR8Qxf4pEYK5keyDjipJauYuofYbqMPrq4NHfOMzLfuqJV5ejxE/ur3WmisbEk0xzdUChhu+HYIN73kRh46zJPdRVYYQydR0r71Ao7RsWNp1u4dtPSH241nuonoFWjc+AE0z5AaVz2jr/2fHYQ07F09qDjppT1dvNSntM+QH4RmVWc83f+3C7LWBqzmQBO1yDj4k2AY9BPLeAk3bl1b29HiAwbuLOoccQNnXxDR3MIY3IPtmUfCSP/oe50jztwcQtXNn2w579vrxiy2f2zR5dDBBDCFLW52bqT+4ROT4wkszN7g+P4bpW3ZHMtb4lQO3KRAG9rX8maok/UGtSRPcCSFHGmmfUyOlEtMVXa6F4cwzUBHIZMR0TPJr2Y467AYrANFih7B8RtexGNCuK5rejc+i+VHSHWwmQQly+iYW6FVkaOesg54uECe/njs+v7QKAJIK/PpKnBeqyBfWoOMaxA5lhfWkSzNO2RxI93s1jQfilelHAlGfnnmJwCW3ME1NKm2zB3R5sOLpve8a84GuTmrL425xr5taCMOPsIN5Ke7Ry3bdzAg02+F2NXzP/29D0S5KyWu1VJA94N9bPhDcyz+1uzZihJu7oHEnYeS7RqK0vNvbpl90yVSYAkfLbhtmEocrBkUJpKJfeYK0UeL+gNO0nih8/dsaDPBUlgoH3eX7QvlmD3NeqKGenuFE+9YhAg9A6+HgXZFq/FsFP72p2/hqHdyQyxzmJ4YAPTNKq+j/GcAjDM9aZ0Px1RYnG0G8uo2zZDyQbCMTqws7ae/NCuK5/itcar9S9tNmpTTn6X20KBxWoD3i7qeQ+vHyhncB9QcY+ZNkTDUXv+Z6POb2lZZiM5A25YxNllNYWeHd/ZaM3t5aOitVDEpQ5o08XFNpDOuOGYL2fUE/Lx4sSWfI4M3zHiQxs6IIDOseCjw3I8BjgC/qOBZ44YJPW5Qi/jgEIDOwem71BUXU9L+2I5WI8ynTUVxQwyMiSfSFNrHox02XVf/WTAIMI1Q3+Hnzo6DZ+xAnARRvvkddk/9j0a29ZJdMILgLweJDyvXU7GgPufgS44bFRGYMOvj7sMNEeHxVCl2LMMzDZI5hD702vuwRHHhAH0OQLMktm2s6ejzOiU+CtQnU6NmE3N2tlnHHC+Oe2Decx58KFg21x7VVmyAY8t/9G22OKzWX5m2/k3nBjd2sY+PmHkZarzAd5FL1aPzsYpF9ZJ4zopmrwrsjXugxt204WYoHJN1a264qpbbLRqbhEMVvkl9x7+CbjhX3/47l/i9//+X9bT93+04uWq3IAMy+mPmdq0ctB71Lnnp9sAem4O+Z+9t5TTB/vBturjp169sDknFZ9V6gWWfR2LNTKUtzVMjVUYS4geyN8z61/z6Pgjd3tJ3V4u57NX6Se3XyRD4+T8oT1y6PLAli2CqWl1gtH8yFFIVLIF16YN7qXjAgYux9IzAzvHw4/DCIog5qMBAQ4gf7cNAAhh93iOntvLM9lu71H/fZUVDdoYMCSOSYwJdCxUCtb5od4yoW4ls6Z7UP7g/KiTEmtUFdVN1LhW1WA5NNj0dYMvCsreVdVhjwflhmCcgQn75VISrqVRz8MxvGPgZoUUFwrdwk1erkBg+HV86Q5dF8uDl0622z62VaIgM2Zp4GV0wyx5kPe2IPTta2Tb1/nosUVvt2Rd9nbdduSQt7crmPB+yLJttzbHcZw4ZivChmb/3GzLgg8qUP2CDMM+rlWWg1exxtLYBKahuxiwdzjvsyH9cW73sFuB9p0dveDTLdkgEFWzOR95BAup3+qKOTdfRdfHAtdS4Tyw4yXw0bnZBjY1ct7XAg3IVp72J8ZPnLqqs9/kPgRykYXPPkchX156H932XPiLKhY+fTaoI+4dgjBK2Khp/8yChr7yvG341Mtv9w6oDdzP6al2K06pqp6OZcNgxoTHOTEw9vZVEds0Beapn0qQRnFO0pVnXy3pHSndzC6LPaiAzfXMw3MqbuXuwbWeXNPhkjg5EENx5nwE0PBsip4PWNTBuXSue4NW6v7mK7+Du8dLWz3Iu7Io+fCMgSkQqujaX8zn3yxDVSMozwJl4gI888Kh/t2brygCgAXzM2Dy1bytRtr+6s6pCMTFe2VlmiVZ8wxxdfipnX0S4V+eXY5Wk6i99ERQA/mTYjkf1hoF4B1vyxSe1uDSL9rVYFG2gJgiBaTo3rsOui7QrKQ417QJm+XlY7DNNlspMP5qNptNb17PuILUanv12id4KoEJJS2Ll7LINTQ3tODtAgDOESC6H9h2Ba7oo50iRCj0xufIdTFb3kWzLybfCo9cmClgLAFaIs5gUW5ztYnQQuPa93y8T1KLnej+AAxYvosQxGX49rW5C1EW6ywl77RltiTP9oSN2iyntMZgFlz7g8pSXCpRB7qxZTGQ3kXh69dd8mGAS9fPVHxS4eijBrqBdrfr+hUaItzWHDTWHi9KzQJ42mfTG67/P2Z9e42ISgn7glCAwQNczBIBgIUbtJqDBK128SNsafkIzLJuggoXFeSkZ4KK/0s0eijBY5a2MYjYBXoxoRfT6c0tFibTmytqYpQihLug5kk+pv2xBHj8DXP6KqJFCYjqJIjqNAjkBi6ChZiW1hbcEEOujlpHsud/wntT4MbAJDVw5HMJ9mJVgscpSThTWPWogsdV1NtcptntKkfSqMtHdbW6fFxxykMFGlBDPegH+N8K47A1X1Qi9edpdanVlV5d6pU/vTH8W1Ud57JGG1FjlzCCxq84CEEhoRn9d6C/vh5kysheR6DbaR24WB98wLUv5ssaMgulJOYDmTXnl0OmqbhOtFmKKcuizvgSAggjGOHsVy30wrg4zzkjjlZ3dcjy9CA3tYB1m8gjhLG8n6v7aWX+lYfP3vUE5/Hl/lV1bE9ng5haquUDQin46DLDWvXe9Mzeh4t8GgMB4nS1hndADa0ss+8SFw46rHyqdmc7S3lQeAhcmq+LYXgMHsxAJwviHoLjjPJ1uX4A35NPM6grAZ+ipoNdE9uJU9Xg9/Du42/faBS+rByrqlLPNTk4AZm8WuRVLh/0w8pb4k6xBfyqV9K+tG6wiEXkof6LqC0Jqcck5j34MM8SzZobUJQma+s8pVoOOPZXXYHO8saOWkAcBwrv1U13O4qVk9QMFA5dnKDl9VE38s+H+yLtX+iYipYY1stLtcGgYP5cWohXSbNPC7QHVEQ/SAWRugxWxt22LlUsA7uN1tulr4iSCzVfLaVGSI3iMl9NTitItAP9uyn21TNr5y18O3vBK5bLXOb38rYod+iDi/ru4dPdaNEK9La5fAlK6jp8LYnPwbWL5fSVP7hSZd1I8hYy27I78ju+i7GkJUZnLz7Z/0ipMBUE/IQXHMjjFT+2959uZpZDj+F/ozADjX4Q6Ghnq/I1f0VASnrM5d9D094BpFu6MEKrZOvodGNlAZoq2/XEdUgioo+xFAMKLRKWGrpr1jaTLm0t+DuzVz0WucJ5R+lD1clXbOVaIP4dm/yedCGI5cVRuscsY/xiH3IlJwSpKAe2sGiIJE6a1YmqUktwyFNEBvJsn4Y1ATOm+b0MLE8nEbdmjDa+fR3taf60eIUnaKiClz2BFfOCyY2a3E2LA/4CphdLQQNxIGT3wWIkUpbl1rCvB7qiSJmgDC8EltSv0I/EDuFQ2PvMOFci+Yzo4O3E3gyjC9RY65FOWc0HbE606iZbLlaTMYYZHTbac1yNET2tncW0Sw9Dc/ttBAmpinzlfw5Hyr99HpXxZqmufNVj6fEsw/CyDqnOfVTto5PRDzdhZaA7dz+gZbcr98TrUVQ/M9lUJfCX+YyJfbqaf/lp9m9d018hptxvsRSD7vFw1D1YY1ZA5J+lHJiV+bP057lvwUd7yIC9k21ZoVf9LNXTWbHp4VDreb4PYGEc2DMv+IGE+biFlrKjyvnrm9nskpzovNzceNV+gSgtp7n8IPfRPJAGM11Gjo9Jpk0pK6XQo8zofqN2I3lmN5I9KOQSdmMixU3oEHLD6sR9R/I/CbEIUSZvkx+rPSNsHERGlP6egLVTBR+lIpEjj9BYnHBLl86dM3lF4fmqNl1Pu6ZL/92bUxc2e/9QodJndob3MnAaegOxCCYfTqDAxamCQyxVY+mZ7KDU0v7EH5dZQ5heO2lJc2vMQmFtBYvLtMl2q0qHznssfZ3ssdazKwrfHDB90mjwWbeq1hZfh5bBIZfJXIYlqnWJEitAA02F+RhAWW5/iIOAZ6EROzxB+Pb1FIfxcRH8YE68Du1ohodf4jiIZGh2d07/BS6m+LHCWKsU8NvxQeucuuJO0jMlbdw5/j154dYxAi40TDO1wQsJWVLTXYlb/IyROjQlyBfIcpKrbEdfrkApdlSSHCqVPAcOkxcJV5NUJ8904NsvNw7dQfSCVyF+z2Fn/xSTTm6xxHX0NHPsGM8bP/obKVxiqwH+8q05F7J958+e4gGix0dkI0d6o0dm4jsBjPGqx/Mnf2PRmpxc22doXQFdW6fVS0vD+9GENBPGSIV1d4SO1v2WXKYHAeJ3dlhoNoIZQ6euiUHlrlR7MDJSrSlDpCt/Oau+ba/0OFljDnIU0HELgs+zuNbWBnIedjqsPr65Jafka/5akqmbXrsfW0b8xBcXeAYyFvVhxVe4P3z31/jmK3cchrvLnugGKo8PX1964deXNnG5XvFK7micAY3sWVPBb+Es2oUF7Qw2l/7OS0p9AjDufnsHKc9Wu/0r8K01DllR2/Xbm/vxsdDVHbmhRNhgjICxKu3YQhSjhQbneCRswreLYRpoOQgrj/eQQoqPNLgnTtaGik8qKdFXDHJ4M4ffXsnb5bmdajefhZnRq0EZ2dzWpUE6QTK5kL7yCYzkDb8mIV+S8EaSDmfO6qwPouhi02zJkn0T9sH08io24CGMkesGfayRE0a0mdxH6FmzjzxUKhIDu4AnNjQAu2kV/AysIRpM8jjwYzziyXUJw2Nkxg5COxhmIJMpsKu9GDl3blsg2MXTNbNWEen/VzbxDjPdIKRPgVg3Co8J4U4afsdL/82ie9oIxax/Y9aup4RbigjMNTK5NdZahPFLXG2UE+gnpiKlEiNLBkd8n7b2OBpR1xYDmcs/J3W7WLERfmZvkfmVMeulqrlJ3CH8QJNJLO4j8ynH8NtqQzh+wKfKS8FWVxmV80ZxnJZJHPu3wBcReJtpGiOHUMcaezYR8OwOooIUp7c8aeuyH4ygkTzMc7lOki4lULuS+T13MmGL4wakuLBWYgB1fFSJJdynhnQ3B4fDzLfwjDAxCHKxA/nCZvR2dmJOTpGdHfvmzNwYu50dHL7+5sTMeC3SQvfMJPbHTMCVzhJM1534ZMmynVk+e3IaLI+YtFURHei2PALCgww/Z+qiosR0Qq6edSXN8mDNOPgQxOisbbnDhKshhhRAm4USQy6IS7eiYT50OAOR7p4TMmRLHAp8fjQtcet55jLeM0B4YTQJmIJjDMqyx6cGffy86NjUUrcCjgCV8iNoVG9AT+0ezczJZ8B5ZOK2yJUnx4/atXv2Zha8nQXXN7PlUNSO4bBlGuNWOSCDzjXoCSIpjqy7K1X4FIquiPhSkWtfw6wRjPnGBnWWoiH6DRIeCAT+vJKIDjf209XcZp84yVC8Tmxpd2q07ukOr4LS++F1y8EXOahPv80uPB/gzPtjVWaeiUFpbBdXdgWVXSEmwYs/Uk+zxRyN3X5J7SWnnboispPFY9TxqHaMWu3Ssd9Snmvt+WdqiLtKMt4PLCc7XyscnMVASs+Yw+ggty0r6xGyrRqjVnnoG/njqrHju8n/mDKxv1d52BiXosbsn3BaBJjLplm15N22jLAsJVrjgEcF8Wn+po4C3ErbDHpx6az1rWABbD4WTOVasOSYfKc4BnTiGN2eOMbPq6H7c/F/Lnr3rcVbAAA="
  },
  "electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
